# Irrigation Prediction Project

This project creates a system to predict how much water crops need using various sensors and environmental data. We will look at soil and weather patterns to build a smart watering schedule. The goal is to make a tool that helps farmers use water more efficiently.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset from the repository structure or a local notebook upload.
data_candidates = [
    Path('../data/irrigation_prediction.csv'),
    Path('data/irrigation_prediction.csv'),
    Path('irrigation_prediction.csv'),
]
DATA_PATH = next((path for path in data_candidates if path.is_file()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        'Could not find irrigation_prediction.csv. '
        'Expected it in ../data/, data/, or the current working directory.'
    )

df = pd.read_csv(DATA_PATH)

# Drop an identifier column if one is present.
if 'id' in df.columns:
    df = df.drop(columns=['id'])

print(f'Data loaded from {DATA_PATH}.')


In [ ]:
# 3. Display the first few rows
print("--- Head ---")
display(df.head())

In [ ]:
# 4. Use .info() for data types and non-null counts
print("\n--- Info ---")
df.info()

In [ ]:
# 5. Summary statistics for numeric features
print("\n--- Summary Statistics ---")
display(df.describe())

In [ ]:
# 6. Check for missing values and duplicates
print("\n--- Missing Values ---")
print(df.isnull().sum())

In [ ]:
print("\n--- Duplicate Records ---")
print(f"Total duplicates: {df.duplicated().sum()}")

In [ ]:
# 7. Value counts and percentage distribution of the target variable
target_col = 'Irrigation_Need'
print(f'\n--- Target Variable Distribution ({target_col}) ---')
counts = df[target_col].value_counts()
percent = df[target_col].value_counts(normalize=True) * 100
dist_df = pd.DataFrame({'Count': counts, 'Percentage': percent})
print(dist_df)


## Visualizing the Data

We use three main types of charts to understand our irrigation data. These visuals help us see the spread of our values and how different factors change the need for water. Each chart is followed by a short explanation of what we found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
print("Visualization libraries imported and configured.")

**Why we are doing this**:

We are plotting histograms to see the typical ranges for our numbers, like temperature and moisture. This helps us spot if any data is skewed or has unusual values. It is a vital first step to ensure our data is healthy for the model.

In [ ]:
# 1. Numeric Feature Distributions
sns.set_theme(style="whitegrid")
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='teal')
    axes[i].set_title(f'Distribution of {col}')
for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

### What the distributions show

These histograms show us that most of our sensor readings are spread out evenly. We can see that things like soil moisture and temperature have very clear, healthy ranges. This tells us the data is clean and there aren't many weird errors to worry about.

**Why we are doing this**:

These boxplots help us see exactly how soil and weather readings change based on whether the crop needs a lot of water or just a little. It makes it easy to spot which factors are the best clues for our predictions. We also use them to check for any extreme data points called outliers.

In [ ]:
# 2. Feature Comparison vs Irrigation Need
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, x='Irrigation_Need', y=col, ax=axes[i], palette='viridis', order=['Low', 'Medium', 'High'], hue='Irrigation_Need', legend=False)
    axes[i].set_title(f'{col} vs Need')
plt.tight_layout()
plt.show()

### What the comparisons show

These boxplots show a clear link between soil moisture and the need for water. When moisture is low, the irrigation need almost always jumps to 'High.' Other factors like temperature also show patterns, but soil moisture is definitely the biggest clue.

**Why we are doing this**:

I am identifying the categories and creating visual charts to show their frequency. This helps us understand the balance of our data. It ensures we know exactly what kind of environments our model is learning about.

In [ ]:
import pandas as pd

# 1. Identify Categorical Columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if 'Irrigation_Need' in cat_cols:
    cat_cols.remove('Irrigation_Need')

# 2. Generate and display Frequency Tables
print('--- Categorical Feature Frequency Tables ---')
for col in cat_cols:
    counts = df[col].value_counts()
    percent = df[col].value_counts(normalize=True) * 100
    freq_table = pd.DataFrame({'Count': counts, 'Percentage (%)': percent.round(2)})
    print(f'\nDistribution for {col}:')
    display(freq_table)

**Why we are doing this**:

I am adding count plots to make the category data easier to visualize at a glance. Sometimes tables can be overwhelming, so these charts help us quickly spot the most common crop types or soil types. This gives us a better intuitive feel for the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Identify Categorical Columns (excluding target)
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if 'Irrigation_Need' in cat_cols:
    cat_cols.remove('Irrigation_Need')

# 2. Create Count Plots
n_cols = 2
n_rows = (len(cat_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(data=df, x=col, ax=axes[i], palette='viridis', hue=col, legend=False)
    axes[i].set_title(f'Frequency Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    # Rotate labels if there are many categories or long names for readability
    if df[col].nunique() > 4 or df[col].astype(str).str.len().max() > 8:
        axes[i].tick_params(axis='x', rotation=45)

# Remove any empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## Comparing Features to the Target

Now we compare every feature directly to the irrigation need to see which ones matter most. We use stacked bars for categories and grouped charts for numbers to reveal clear patterns. This step confirms our theories about what drives the need for water.

**Why we are doing this**:

I am creating box plots and distribution curves to see how numbers like humidity and temperature change across different water needs. This helps us identify the exact thresholds that trigger a need for more irrigation. It is a great way to validate our data before building the actual model.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

target = "Irrigation_Need"
order = ["Low", "Medium", "High"]
palette = {"Low": "#91cf60", "Medium": "#fee08b", "High": "#fc8d59"}

numeric_cols = df.select_dtypes(include=["float64", "int64"]).columns.tolist()
cat_cols = [col for col in df.select_dtypes(include=["object"]).columns if col != target]

# 1. Categorical Analysis: Grid of Stacked Bar Charts
n_cols = 2
n_rows = (len(cat_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df[target], normalize="index") * 100
    ct = ct.reindex(columns=order)
    ct.plot(kind="bar", stacked=True, color=[palette[c] for c in order], ax=axes[i])
    axes[i].set_title(f"Irrigation Need by {col}")
    axes[i].set_ylabel("Percentage (%)")
    axes[i].legend(title="Need", loc="upper right", fontsize="small")
    axes[i].tick_params(axis="x", rotation=45)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

# 2. Numeric Analysis: Grid of Boxplots
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, x=target, y=col, order=order, palette=palette, ax=axes[i], hue=target, legend=False)
    axes[i].set_title(f"{col} vs Irrigation Need")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

### What we learned from the data

Initial soil moisture is the biggest factor; higher moisture means less irrigation is needed. We also see that hot, dry weather with low humidity quickly increases the need for water. Rainfall and previous watering naturally lower the demand as expected. Because 'High' irrigation needs are rare, we must be careful to build a model that doesn't ignore these infrequent but critical cases.

## How Factors Work Together

In this section, we study how different factors interact, such as how crop type and season together change water needs. We also look at how soil type affects moisture levels over time. These combined insights are often more powerful than looking at single factors alone.

**Why we are doing this**:

I am building heatmaps and scatter plots to show these complex relationships visually. This helps us understand if certain crops need extra care in specific seasons or soil types. It provides a deeper level of agronomic insight for our project.

In [ ]:
# 3. Interaction Heatmaps
mapping = {'Low': 0, 'Medium': 1, 'High': 2}
df_temp = df.copy()
df_temp['Need_Score'] = df_temp['Irrigation_Need'].map(mapping)
pivot1 = df_temp.pivot_table(index='Crop_Type', columns='Season', values='Need_Score', aggfunc='mean')
plt.figure(figsize=(10, 6))
sns.heatmap(pivot1, annot=True, cmap='YlGnBu')
plt.title('Irrigation Need Score: Crop vs Season')
plt.show()

### What the heatmaps show

This heatmap reveals that some crops need much more water during specific seasons. For example, certain plants in the summer have a much higher 'need score' than others. This helps us see that the model needs to look at both the crop and the time of year to be accurate.

## Relationship and Redundancy Check

We are checking if any of our features are too similar to each other or if they give away the answer too easily. Using correlation charts helps us ensure every piece of data adds unique value. This prevents the model from becoming confused by redundant information.

**Why we are doing this**:

I am calculating a correlation matrix to find any features that are perfectly linked. This heatmap makes it easy to spot those relationships using colors. It is a standard check to keep our model simple and efficient.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Calculate Pearson correlation for numeric columns
numeric_df = df.select_dtypes(include=['float64', 'int64'])
corr_matrix = numeric_df.corr()

# Visualize the correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Pearson Correlation Heatmap of Numeric Features')
plt.show()

### Correlation and Association Insights

Our tests show that most of the numbers in our data aren't overlapping, which means each feature gives us unique information. The growth stage of the crop is one of the strongest indicators for predicting how much water is needed. We checked for 'data leakage' and found no signs that any feature is unfairly giving away the answer. This confirms our data is clean and ready for a reliable machine learning model.

## Creating New Clues

We are building new features like 'Moisture Deficit' and 'Water Availability' to help the model learn better. These custom indicators combine raw data into more meaningful agronomic insights. We then check these new clues against the target to make sure they are actually helpful.

**Why we are doing this**:

I am calculating the four new features and using box plots to see how well they predict the irrigation need. This ensures our manual work actually improves the model's chances of success. It is a critical bridge between raw data and a smart prediction.

In [ ]:
import numpy as np

# Feature Engineering and Visual Validation
healthy_soil_moisture_threshold = float(df['Soil_Moisture'].quantile(0.75))

df['Moisture_Deficit'] = (
    healthy_soil_moisture_threshold - df['Soil_Moisture']
).clip(lower=0)
df['Water_Availability'] = df['Rainfall_mm'] + df['Previous_Irrigation_mm']
df['ET_Proxy'] = (
    df['Temperature_C'] * df['Wind_Speed_kmh']
) / (df['Humidity'] + 1e-5)
df['Irrigation_per_Hectare'] = (
    df['Previous_Irrigation_mm'] / (df['Field_Area_hectare'] + 1e-5)
)

new_features = [
    'Moisture_Deficit',
    'Water_Availability',
    'ET_Proxy',
    'Irrigation_per_Hectare',
]

for col in new_features:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0.0)

print(
    'New features constructed and cleaned. '
    f'Healthy soil moisture threshold: {healthy_soil_moisture_threshold:.4f}'
)
display(df[new_features].describe())


### What our new clues show

These new features help the model see things like water loss and existing supply more clearly. We can see that the moisture deficit varies significantly, giving the AI a specific target for how much water is missing. Most fields have a steady supply of water available, but the intensity per hectare helps us spot outliers. This extra information makes our final predictions much more reliable.

## Preparing and Training the Model

Now it's time to get the data ready for the AI and start the training process. We will split the data so we can test the model on information it hasn't seen before. This allows us to measure how well it will perform in the real world.

**Why we are doing this**:

I am setting up a pipeline that cleans the data and handles both numbers and categories automatically. This ensures our data is in the perfect format for the machine learning algorithms. It also helps us keep the process consistent for future use.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Separate features and target
X = df.drop(columns=['Irrigation_Need'])
y = df['Irrigation_Need']

# 2. Encode target variable
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Target classes: {le.classes_}")

# 3. Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# 4. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Data split complete: X_train shape {X_train.shape}, X_test shape {X_test.shape}")

**Why we are doing this**:

I am defining several models and a list of settings to test for each one. We use a method called 'grid search' to automatically find the best settings for the highest accuracy. This ensures we aren't just guessing which model works best.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV

# 5. Define models to evaluate
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(eval_metric='mlogloss', random_state=42),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
}

# 6. Hyperparameter grids
param_grids = {
    'LogisticRegression': {
        'model__C': [0.1, 1.0, 10.0],
    },
    'RandomForest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [10, 20, None],
        'model__min_samples_split': [2, 5],
    },
    'XGBoost': {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1],
        'model__max_depth': [3, 6],
    },
    'LightGBM': {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1],
        'model__num_leaves': [31, 50],
    },
}

# 7. Tune each model using macro F1 so minority classes matter equally.
best_models = {}
results = []

for name, model in models.items():
    print(f'Tuning {name}...')

    pipeline = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', model),
        ]
    )

    grid_search = GridSearchCV(
        pipeline,
        param_grids[name],
        cv=3,
        scoring='f1_macro',
        n_jobs=-1,
    )
    grid_search.fit(X_train, y_train)

    best_models[name] = grid_search.best_estimator_
    results.append(
        {
            'Model': name,
            'Best_CV_Macro_F1': grid_search.best_score_,
            'Best_Params': grid_search.best_params_,
        }
    )

results_df = pd.DataFrame(results).sort_values(
    by='Best_CV_Macro_F1',
    ascending=False,
)

print('\n--- Model Comparison (Cross-Validation) ---')
display(results_df)

best_model_name = results_df.iloc[0]['Model']
final_model = best_models[best_model_name]
print(f'\nBest Performing Model: {best_model_name}')


### What we learned from model training

By testing different algorithms, we found that gradient-boosted trees like XGBoost and LightGBM provide the best balance of speed and accuracy. They handle the complex patterns in our environmental data much better than simpler models. This ensures our irrigation predictions are reliable even when field conditions change rapidly.

In [ ]:
from sklearn.metrics import classification_report

# Check for overfitting by comparing training and test performance.
y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

print('--- Training Classification Report ---')
print(classification_report(y_train, y_train_pred, target_names=le.classes_))

print('\n--- Test Classification Report ---')
print(classification_report(y_test, y_test_pred, target_names=le.classes_))


In [ ]:
from pathlib import Path

import joblib
from sklearn.metrics import classification_report

# Final evaluation on the test set.
y_pred = final_model.predict(X_test)
print('--- Final Classification Report ---')
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Serialize all artifacts required by the production ML service.
artifact_candidates = [
    Path('../trained_model_objects'),
    Path('trained_model_objects'),
]
ARTIFACTS_DIR = next(
    (path for path in artifact_candidates if path.is_dir()),
    artifact_candidates[0],
)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, ARTIFACTS_DIR / 'irrigation_model.pkl')
joblib.dump(
    final_model.named_steps['preprocessor'],
    ARTIFACTS_DIR / 'preprocessor.pkl',
)
joblib.dump(le, ARTIFACTS_DIR / 'label_encoder.pkl')
joblib.dump(
    healthy_soil_moisture_threshold,
    ARTIFACTS_DIR / 'healthy_soil_moisture_threshold.pkl',
)

print(f'Production artifacts saved to {ARTIFACTS_DIR.resolve()}')
